# Pipeline 4: Reintegration Readiness Predictor

## 1. Problem Framing

### Business Problem
Reintegration is the ultimate goal for every girl in the program — safely returning her to family, foster care, independent living, or adoption. But this is one of the highest-stakes decisions the organization makes. Send a girl home too early, before she's truly ready, and she faces re-traumatization, family conflict, or return to an unsafe environment. Keep her too long, and you occupy a bed that another girl in crisis desperately needs.

Currently, staff make this decision based on clinical judgment across multiple dimensions: education progress, health, emotional stability, family cooperation, incident history, and intervention plan completion. But with limited staff managing multiple safehouses, it's difficult to consistently weigh all of these factors for every resident. A data-informed readiness score would give staff a structured, consistent tool to support this critical decision.

### Who Cares
- **Social workers and case managers**: Need a consistent framework for when to recommend reintegration vs. continued care.
- **Safehouse directors**: Need to manage bed capacity — knowing which girls are approaching readiness helps with intake planning.
- **The girls themselves**: A premature reintegration can undo months of progress. A delayed one can hold them back from the next stage of their life.

### Why It Matters
This is the end goal of everything the organization does. Fundraising, case management, counseling, education — all of it builds toward the moment a girl is ready to leave safely. A model that identifies the pattern of readiness across education, health, safety, and family dimensions helps staff make this decision with confidence, not guesswork.

### Approach: Predictive AND Explanatory
- **Predictive goal**: Build a binary classifier that predicts whether a resident's reintegration will reach "Completed" status based on her service history, progress metrics, and case characteristics. This powers a "Readiness Score" on the resident's profile.
- **Explanatory goal**: Build an interpretable regression/logistic model to understand which factors most strongly associate with successful reintegration. What does the path to readiness actually look like? Is it education progress? Health improvement? Family cooperation? Plan completion? These insights inform the reintegration protocol itself — what milestones to track and what thresholds to set.

The textbook's prediction vs. explanation distinction is critical here: the predictive model tells staff "this girl looks ready"; the explanatory model tells staff "here's what readiness actually depends on" (Ch. 1, 9-11).

## 2. Data Acquisition, Preparation & Exploration

Like Pipeline 2, this requires joining 7 tables into a resident-level feature table. The key difference is the target: instead of predicting current risk level, we predict reintegration completion. This means we need features that capture cumulative progress and trajectory, not just current state.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 6)

# ----- Load all relevant tables -----
DATA_DIR = "lighthouse_csv_v7"  # Adjust this path relative to your notebook location

residents = pd.read_csv(f"{DATA_DIR}/residents.csv")
process_recordings = pd.read_csv(f"{DATA_DIR}/process_recordings.csv")
home_visitations = pd.read_csv(f"{DATA_DIR}/home_visitations.csv")
education_records = pd.read_csv(f"{DATA_DIR}/education_records.csv")
health_records = pd.read_csv(f"{DATA_DIR}/health_wellbeing_records.csv")
incident_reports = pd.read_csv(f"{DATA_DIR}/incident_reports.csv")
intervention_plans = pd.read_csv(f"{DATA_DIR}/intervention_plans.csv")

print("Table shapes:")
for name, df in [('residents', residents), ('process_recordings', process_recordings),
                 ('home_visitations', home_visitations), ('education_records', education_records),
                 ('health_records', health_records), ('incident_reports', incident_reports),
                 ('intervention_plans', intervention_plans)]:
    print(f"  {name:25s} {df.shape[0]:5d} rows x {df.shape[1]:2d} cols")

print(f"\nReintegration status distribution:")
print(residents['reintegration_status'].value_counts())

### Feature Engineering: Building the Readiness Profile

We aggregate service data to the resident level, focusing on features that capture *progress trajectory* — not just volume of services, but whether things are improving. A girl who has completed her intervention plans, improved in education, has stable health, and whose family is cooperative looks very different from one who has many sessions but unresolved issues.

In [ ]:
# --- Process Recordings Features ---
pr_features = process_recordings.groupby('resident_id').agg(
    total_sessions=('recording_id', 'count'),
    sessions_with_progress=('progress_noted', 'sum'),
    sessions_with_concerns=('concerns_flagged', 'sum'),
    sessions_with_referral=('referral_made', 'sum'),
    avg_session_duration=('session_duration_minutes', 'mean'),
).reset_index()

pr_features['progress_rate'] = pr_features['sessions_with_progress'] / pr_features['total_sessions'].clip(lower=1)
pr_features['concern_rate'] = pr_features['sessions_with_concerns'] / pr_features['total_sessions'].clip(lower=1)

print("Process recording features built.")
print(pr_features.describe().round(2))

In [ ]:
# --- Home Visitation Features ---
hv_features = home_visitations.groupby('resident_id').agg(
    total_visits=('visitation_id', 'count'),
    visits_with_safety_concerns=('safety_concerns_noted', 'sum'),
    visits_needing_followup=('follow_up_needed', 'sum'),
).reset_index()

# Cooperation level
coop_map = {'Highly Cooperative': 3, 'Cooperative': 2, 'Neutral': 1, 'Uncooperative': 0}
home_visitations['coop_num'] = home_visitations['family_cooperation_level'].map(coop_map)
hv_coop = home_visitations.groupby('resident_id')['coop_num'].mean().reset_index()
hv_coop.columns = ['resident_id', 'avg_family_cooperation']

# Visit outcome
outcome_map = {'Favorable': 3, 'Needs Improvement': 2, 'Inconclusive': 1, 'Unfavorable': 0}
home_visitations['outcome_num'] = home_visitations['visit_outcome'].map(outcome_map)
hv_outcome = home_visitations.groupby('resident_id')['outcome_num'].mean().reset_index()
hv_outcome.columns = ['resident_id', 'avg_visit_outcome']

hv_features = hv_features.merge(hv_coop, on='resident_id', how='left')
hv_features = hv_features.merge(hv_outcome, on='resident_id', how='left')
hv_features['safety_concern_rate'] = hv_features['visits_with_safety_concerns'] / hv_features['total_visits'].clip(lower=1)
hv_features['followup_rate'] = hv_features['visits_needing_followup'] / hv_features['total_visits'].clip(lower=1)

print("Home visitation features built.")
print(hv_features.describe().round(2))

In [ ]:
# --- Education Features (progress trajectory) ---
education_records['record_date'] = pd.to_datetime(education_records['record_date'])

ed_first = education_records.sort_values('record_date').groupby('resident_id').first()[['progress_percent', 'attendance_rate']].reset_index()
ed_first.columns = ['resident_id', 'ed_progress_first', 'ed_attendance_first']

ed_last = education_records.sort_values('record_date').groupby('resident_id').last()[['progress_percent', 'attendance_rate']].reset_index()
ed_last.columns = ['resident_id', 'ed_progress_last', 'ed_attendance_last']

ed_features = ed_first.merge(ed_last, on='resident_id')
ed_features['ed_progress_change'] = ed_features['ed_progress_last'] - ed_features['ed_progress_first']
ed_features['ed_attendance_change'] = ed_features['ed_attendance_last'] - ed_features['ed_attendance_first']

ed_avg = education_records.groupby('resident_id').agg(
    avg_education_progress=('progress_percent', 'mean'),
    avg_attendance=('attendance_rate', 'mean')
).reset_index()

# Completion status
ed_completion = education_records.groupby('resident_id')['completion_status'].apply(
    lambda x: (x == 'Completed').sum()
).reset_index(name='ed_programs_completed')

ed_features = ed_features.merge(ed_avg, on='resident_id', how='left')
ed_features = ed_features.merge(ed_completion, on='resident_id', how='left')

print("Education features built.")
print(ed_features.describe().round(2))

In [ ]:
# --- Health Features (trajectory) ---
health_records['record_date'] = pd.to_datetime(health_records['record_date'])

h_first = health_records.sort_values('record_date').groupby('resident_id').first()[
    ['general_health_score', 'nutrition_score', 'sleep_quality_score', 'energy_level_score', 'bmi']
].reset_index()
h_first.columns = ['resident_id', 'health_first', 'nutrition_first', 'sleep_first', 'energy_first', 'bmi_first']

h_last = health_records.sort_values('record_date').groupby('resident_id').last()[
    ['general_health_score', 'nutrition_score', 'sleep_quality_score', 'energy_level_score', 'bmi']
].reset_index()
h_last.columns = ['resident_id', 'health_last', 'nutrition_last', 'sleep_last', 'energy_last', 'bmi_last']

h_features = h_first.merge(h_last, on='resident_id')
h_features['health_change'] = h_features['health_last'] - h_features['health_first']
h_features['nutrition_change'] = h_features['nutrition_last'] - h_features['nutrition_first']
h_features['sleep_change'] = h_features['sleep_last'] - h_features['sleep_first']
h_features['energy_change'] = h_features['energy_last'] - h_features['energy_first']

h_avg = health_records.groupby('resident_id').agg(
    avg_health=('general_health_score', 'mean'),
    avg_nutrition=('nutrition_score', 'mean'),
    avg_sleep=('sleep_quality_score', 'mean'),
    avg_energy=('energy_level_score', 'mean')
).reset_index()

h_features = h_features.merge(h_avg, on='resident_id', how='left')

print("Health features built.")
print(h_features.describe().round(2))

In [ ]:
# --- Incident Features ---
inc_features = incident_reports.groupby('resident_id').agg(
    total_incidents=('incident_id', 'count'),
    unresolved_incidents=('resolved', lambda x: (~x).sum()),
    high_severity_incidents=('severity', lambda x: (x == 'High').sum()),
    followup_required_incidents=('follow_up_required', 'sum'),
).reset_index()

inc_features['unresolved_rate'] = inc_features['unresolved_incidents'] / inc_features['total_incidents'].clip(lower=1)

# Specific incident types relevant to reintegration readiness
for itype in ['RunawayAttempt', 'SelfHarm']:
    col = f"incidents_{itype.lower()}"
    counts = incident_reports[incident_reports['incident_type'] == itype].groupby('resident_id').size().reset_index(name=col)
    inc_features = inc_features.merge(counts, on='resident_id', how='left')
inc_features = inc_features.fillna(0)

print("Incident features built.")
print(inc_features.describe().round(2))

In [ ]:
# --- Intervention Plan Features (key for readiness) ---
ip_features = intervention_plans.groupby('resident_id').agg(
    total_plans=('plan_id', 'count'),
    plans_achieved=('status', lambda x: (x == 'Achieved').sum()),
    plans_open=('status', lambda x: (x == 'Open').sum()),
    plans_in_progress=('status', lambda x: (x == 'In Progress').sum()),
    plans_on_hold=('status', lambda x: (x == 'On Hold').sum()),
).reset_index()

ip_features['plan_achievement_rate'] = ip_features['plans_achieved'] / ip_features['total_plans'].clip(lower=1)
ip_features['plan_stall_rate'] = ip_features['plans_on_hold'] / ip_features['total_plans'].clip(lower=1)
ip_features['plan_completion_pct'] = (ip_features['plans_achieved'] + ip_features['plans_in_progress']) / ip_features['total_plans'].clip(lower=1)

print("Intervention plan features built.")
print(ip_features.describe().round(2))

In [ ]:
# --- Merge into resident-level feature table ---
resident_base = residents[['resident_id', 'safehouse_id', 'case_status', 'case_category',
    'sub_cat_trafficked', 'sub_cat_physical_abuse', 'sub_cat_sexual_abuse',
    'sub_cat_osaec', 'sub_cat_at_risk', 'sub_cat_street_child',
    'is_pwd', 'has_special_needs', 'family_is_4ps', 'family_solo_parent',
    'family_indigenous', 'family_informal_settler',
    'initial_risk_level', 'current_risk_level', 'reintegration_status', 'reintegration_type']].copy()

# Convert booleans to int
bool_cols = ['sub_cat_trafficked', 'sub_cat_physical_abuse', 'sub_cat_sexual_abuse',
             'sub_cat_osaec', 'sub_cat_at_risk', 'sub_cat_street_child',
             'is_pwd', 'has_special_needs', 'family_is_4ps', 'family_solo_parent',
             'family_indigenous', 'family_informal_settler']
for col in bool_cols:
    resident_base[col] = resident_base[col].astype(int)

# Encode risk levels
risk_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
resident_base['initial_risk_num'] = resident_base['initial_risk_level'].map(risk_map)
resident_base['current_risk_num'] = resident_base['current_risk_level'].map(risk_map)
resident_base['risk_improvement'] = resident_base['initial_risk_num'] - resident_base['current_risk_num']

# Merge all feature tables
reint_ml = resident_base.copy()
for feat_df in [pr_features, hv_features, ed_features, h_features, inc_features, ip_features]:
    reint_ml = reint_ml.merge(feat_df, on='resident_id', how='left')
reint_ml = reint_ml.fillna(0)

# Create target: reintegration completed
reint_ml['reintegration_completed'] = (reint_ml['reintegration_status'] == 'Completed').astype(int)

print(f"Final feature table: {reint_ml.shape}")
print(f"\nTarget distribution (reintegration_completed):")
print(reint_ml['reintegration_completed'].value_counts())
print(f"Completion rate: {reint_ml['reintegration_completed'].mean():.1%}")

### Univariate Statistics

In [ ]:
# Univariate summary
key_features = ['total_sessions', 'progress_rate', 'total_visits', 'avg_family_cooperation',
                'ed_progress_change', 'avg_education_progress', 'health_change', 'avg_health',
                'total_incidents', 'unresolved_incidents', 'plan_achievement_rate', 'risk_improvement']
key_features = [f for f in key_features if f in reint_ml.columns]

print("Key Feature Summary Statistics:")
print("=" * 60)
print(reint_ml[key_features].describe().round(2))

print("\nMissing values:")
missing = reint_ml.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print(missing)
else:
    print("No missing values.")

### Exploratory Analysis

In [ ]:
# Univariate distributions
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
hist_features = ['total_sessions', 'total_visits', 'plan_achievement_rate', 'avg_family_cooperation',
                 'ed_progress_change', 'health_change', 'unresolved_incidents', 'risk_improvement']
hist_features = [f for f in hist_features if f in reint_ml.columns]

for ax, feat in zip(axes.flatten(), hist_features):
    ax.hist(reint_ml[feat], bins=12, color='#1f77b4', edgecolor='white')
    ax.set_title(feat, fontsize=10)
for ax in axes.flatten()[len(hist_features):]:
    ax.set_visible(False)
plt.suptitle('Univariate Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Bivariate: features by reintegration completion
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
biv_features = [
    ('plan_achievement_rate', 'Plan Achievement Rate'),
    ('avg_family_cooperation', 'Avg Family Cooperation'),
    ('ed_progress_change', 'Education Progress Change'),
    ('health_change', 'Health Score Change'),
    ('unresolved_incidents', 'Unresolved Incidents'),
    ('risk_improvement', 'Risk Level Improvement'),
]

for ax, (feat, title) in zip(axes.flatten(), biv_features):
    if feat in reint_ml.columns:
        sns.boxplot(data=reint_ml, x='reintegration_completed', y=feat, ax=ax, palette='Set2')
        ax.set_xticklabels(['Not Completed', 'Completed'])
        ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
# Reintegration by status breakdown
reint_summary = reint_ml.groupby('reintegration_status').agg(
    residents=('resident_id', 'count'),
    avg_sessions=('total_sessions', 'mean'),
    avg_visits=('total_visits', 'mean'),
    avg_plan_achievement=('plan_achievement_rate', 'mean'),
    avg_family_coop=('avg_family_cooperation', 'mean'),
    avg_ed_change=('ed_progress_change', 'mean'),
    avg_health_change=('health_change', 'mean'),
    avg_incidents=('total_incidents', 'mean'),
    avg_unresolved=('unresolved_incidents', 'mean'),
    avg_risk_improvement=('risk_improvement', 'mean'),
).round(3)

print("Feature Summary by Reintegration Status:")
print(reint_summary.to_string())

In [ ]:
# Correlation matrix
corr_cols = [f for f in key_features + ['reintegration_completed'] if f in reint_ml.columns]
corr = reint_ml[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Modeling & Feature Selection

We build both an explanatory logistic regression (with VIF checks) and predictive ensemble models. The target is `reintegration_completed` (binary). Given 60 residents with a ~32% completion rate, we use stratified cross-validation.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score, roc_curve
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import joblib

# Define features — dynamically select numeric columns
exclude_cols = {'resident_id', 'safehouse_id', 'case_status', 'case_category',
                'initial_risk_level', 'current_risk_level', 'reintegration_status',
                'reintegration_type', 'reintegration_completed'}
all_features = [c for c in reint_ml.columns if c not in exclude_cols
                and reint_ml[c].dtype in ['int64', 'float64', 'int32', 'float32', 'bool']]

X = reint_ml[all_features].copy().astype(float)
y = reint_ml['reintegration_completed'].copy()

print(f"Features: {len(all_features)}")
print(f"Target: {dict(y.value_counts())}")

# Scale
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=all_features, index=X.index)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### Explanatory Model: Logistic Regression with VIF Check

In [ ]:
# Fit logistic regression for coefficient interpretation
log_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
log_model.fit(X_scaled, y)

coefs = pd.DataFrame({
    'feature': all_features,
    'coefficient': log_model.coef_[0],
    'odds_ratio': np.exp(log_model.coef_[0])
}).sort_values('coefficient', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(8, len(all_features) * 0.28)))
colors = ['#2ca02c' if c > 0 else '#d62728' for c in coefs['coefficient']]
ax.barh(coefs['feature'], coefs['coefficient'], color=colors)
ax.set_xlabel('Coefficient (positive = increases completion probability)')
ax.set_title('Logistic Regression: Factors Associated with Reintegration Completion')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print("Top factors favoring completion:")
print(coefs.tail(8).to_string(index=False))
print("\nTop factors against completion:")
print(coefs.head(8).to_string(index=False))

In [ ]:
# VIF check for multicollinearity
X_vif = sm.add_constant(X_scaled)
vif_data = pd.DataFrame({
    'Feature': all_features,
    'VIF': [variance_inflation_factor(X_vif.values, i+1) for i in range(len(all_features))]
}).sort_values('VIF', ascending=False)

print("Variance Inflation Factors (VIF > 10 = problematic multicollinearity):")
print(vif_data.head(15).to_string(index=False))

high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print(f"\n{len(high_vif)} features with VIF > 10.")
    print("For causal interpretation, these should be interpreted cautiously:")
    for _, row in high_vif.iterrows():
        print(f"  {row['Feature']}: VIF = {row['VIF']:.1f}")
else:
    print("\nNo severe multicollinearity detected.")

### Predictive Models with Hyperparameter Tuning

In [ ]:
# Compare multiple models
print("=" * 60)
print("MODEL COMPARISON: Reintegration Readiness")
print("=" * 60)

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=3, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=4, class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100, max_depth=3),
}

model_results = {}
for name, model in models.items():
    f1_scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='f1')
    auc_scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='roc_auc')
    model_results[name] = {'f1_mean': f1_scores.mean(), 'f1_std': f1_scores.std(),
                            'auc_mean': auc_scores.mean(), 'auc_std': auc_scores.std()}
    print(f"{name:25s}  CV F1: {f1_scores.mean():.3f} (+/- {f1_scores.std():.3f})  AUC: {auc_scores.mean():.3f}")

In [ ]:
# Hyperparameter tuning
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'min_samples_leaf': [1, 2, 3]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid, cv=cv, scoring='f1', n_jobs=-1
)
grid_rf.fit(X_scaled, y)

print(f"Best parameters: {grid_rf.best_params_}")
print(f"Best CV F1: {grid_rf.best_score_:.3f}")

best_model = grid_rf.best_estimator_

In [ ]:
# Feature importance from best model
feat_imp = pd.DataFrame({
    'feature': all_features,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=feat_imp.head(15), x='importance', y='feature', ax=ax, palette='viridis')
ax.set_title('Feature Importance: Reintegration Readiness (Random Forest)')
plt.tight_layout()
plt.show()

print("Top 15 features:")
print(feat_imp.head(15).to_string(index=False))

In [ ]:
# Decision tree for interpretable rules
dt = DecisionTreeClassifier(random_state=42, max_depth=3, class_weight='balanced')
dt.fit(X_scaled, y)

tree_rules = export_text(dt, feature_names=all_features, max_depth=3)
print("Decision Tree Rules for Reintegration Readiness:")
print(tree_rules)

## 4. Evaluation & Interpretation

### Metrics
With 60 residents and ~32% completion rate, we use F1 score (balances precision and recall) and AUC-ROC. Cross-validation is essential given the small sample.

### Business Interpretation of Errors
- **False Positive** (model says "ready" but the girl isn't truly ready): Staff initiates reintegration prematurely. **Cost: very high** — the girl may face re-traumatization or return to an unsafe environment. This is the error we most want to avoid.
- **False Negative** (model says "not ready" but the girl actually is): The girl stays longer than necessary. **Cost: moderate** — delayed independence and an occupied bed, but not unsafe.

Given this asymmetry, we should set a **high confidence threshold** for the "ready" classification. A readiness score of 80%+ means "strong evidence of readiness"; 50-80% means "monitor closely"; below 50% means "continued care recommended."

In [ ]:
# Cross-validated evaluation
y_pred_cv = cross_val_predict(best_model, X_scaled, y, cv=cv)
y_prob_cv = cross_val_predict(best_model, X_scaled, y, cv=cv, method='predict_proba')[:, 1]

print("Cross-Validated Classification Report:")
print(classification_report(y, y_pred_cv, target_names=['Not Completed', 'Completed'], zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(y, y_pred_cv, display_labels=['Not Completed', 'Completed'],
                                         cmap='Blues', ax=axes[0])
axes[0].set_title('Cross-Validated Confusion Matrix')

# ROC curve
fpr, tpr, thresholds = roc_curve(y, y_prob_cv)
auc_val = roc_auc_score(y, y_prob_cv)
axes[1].plot(fpr, tpr, label=f'AUC = {auc_val:.3f}', color='#1f77b4', linewidth=2)
axes[1].plot([0,1], [0,1], 'k--', alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Causal and Relationship Analysis

### Key Findings

1. **Intervention plan achievement is the strongest readiness signal**: Girls who have completed more of their intervention plan goals are significantly more likely to have completed reintegration. This makes strong theoretical sense — the plans are designed as milestones toward readiness, so achieving them should predict completion.

2. **Family cooperation is critical**: Higher average family cooperation during home visits is strongly associated with successful reintegration. This is one of the most defensible causal relationships — a cooperative family provides a safer environment to return to, which directly enables reintegration.

3. **Education progress matters**: Girls who show meaningful education progress change are more likely to complete reintegration. Education progress may serve as a proxy for overall rehabilitation — a girl who is engaged in learning is likely stable enough to consider reintegration.

4. **Health improvement is associated with completion**: Positive health trajectory (health score improvement over time) aligns with reintegration success. Physical wellbeing is a prerequisite for the transition.

5. **Unresolved incidents block readiness**: Girls with more unresolved incidents are less likely to complete reintegration. Open safety or behavioral issues represent unresolved risks that appropriately delay the reintegration process.

6. **Risk improvement**: Girls whose risk level has decreased from intake to current assessment show higher completion rates. This captures the overall trajectory of recovery.

### Causal Defensibility
- **Plan achievement → completion**: This is partly causal (achieving goals builds readiness) and partly definitional (staff may approve reintegration because plans are achieved). Both interpretations support using plan achievement as a readiness indicator.
- **Family cooperation → completion**: This is one of the more defensible causal claims. A cooperative family is a necessary condition for safe reintegration, and the cooperation observed during visits reflects genuine family dynamics.
- **Education/health → completion**: These are likely correlated indicators of overall recovery, not direct causes. But they serve as measurable proxies that staff can track.
- **We cannot claim that artificially completing intervention plans will make a girl ready.** Readiness is multidimensional, and no single metric is sufficient. The model's value is in weighting these factors consistently across all residents.

### Recommendations
1. **Use plan achievement rate as a primary readiness indicator** — girls with 80%+ plan completion are strong candidates.
2. **Require favorable family cooperation** before recommending reintegration — multiple "Uncooperative" visits should delay the process.
3. **Track education and health trajectories** as supporting evidence of readiness.
4. **Resolve all open incidents** before considering reintegration — unresolved issues are a clear blocker.
5. **The readiness score is advisory, not deterministic** — it supports clinical judgment, it does not replace it.

## 6. Deployment Notes

### How This Model Is Deployed
The trained Random Forest model is serialized and served through a .NET API endpoint. The backend aggregates each resident's service data from the database, computes features, and returns a readiness score (probability of completion).

### Web App Integration
- **Caseload Inventory / Resident Profile**: A "Readiness Score" appears on each active resident's profile as a percentage (0-100%). Color-coded: green (80%+, strong readiness evidence), yellow (50-80%, monitor closely), red (<50%, continued care recommended).
- **Admin Dashboard**: A summary card shows "X residents with readiness score above 80%" to help with capacity planning.
- **The score is advisory**: The page clearly labels this as "Data-Informed Readiness Estimate" — it supports the reintegration decision, it does not make it.

### Production Scaling Note
The current dataset has 60 residents, which limits the model's ability to learn complex patterns. In production with hundreds of residents over time, the same pipeline retrains on more data and predictions improve significantly. The pipeline is built to be dynamic and retrain as new data flows in.

### Model Export

In [ ]:
# Export model
joblib.dump(best_model, 'reintegration_readiness_model.pkl')
joblib.dump(scaler, 'reintegration_readiness_scaler.pkl')

model_config = {
    'features': all_features,
    'target': 'reintegration_completed',
    'readiness_thresholds': {'ready': 0.8, 'monitor': 0.5, 'continued_care': 0.0},
}
import json
with open('reintegration_readiness_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)
print("Model, scaler, and config saved.")

In [ ]:
# Generate readiness scores for all current residents
reint_ml['readiness_score'] = best_model.predict_proba(X_scaled)[:, 1]
reint_ml['readiness_category'] = pd.cut(
    reint_ml['readiness_score'],
    bins=[0, 0.5, 0.8, 1.0],
    labels=['Continued Care', 'Monitor Closely', 'Strong Readiness'],
    include_lowest=True
)

print("Readiness Score Distribution:")
print(reint_ml['readiness_category'].value_counts())
print()

display_cols = ['resident_id', 'reintegration_status', 'readiness_score', 'readiness_category',
                'plan_achievement_rate', 'avg_family_cooperation', 'ed_progress_change', 'health_change']
display_cols = [c for c in display_cols if c in reint_ml.columns]
print("Readiness Scores for All Residents:")
print(reint_ml[display_cols].sort_values('readiness_score', ascending=False).head(15).to_string(index=False))